# Step 1: Load and inspect raw ETF files

We load the three Kaggle price files (SPY, TLT, XLF), convert the `Date` column to real dates using a strict format so any malformed date stops the notebook, and sort each table by date. We then check for repeated dates and record each file's first date, last date and number of rows. These checks confirm the files cover the expected period (25 Feb 2005 – 10 Nov 2017) before anything is built on them.

In [1]:
RAW = 'data/raw/' # relative to the notebook's working folder
import os; print(os.listdir(RAW)) # should list the 4 files

['spy.us.txt', 'tlt.us.txt', 'VIXCLS.csv', 'xlf.us.txt']


In [2]:
import os
import pandas as pd

FILES = {"SPY": "spy.us.txt", "TLT": "tlt.us.txt", "XLF": "xlf.us.txt"}

def load_etf(path):
    df = pd.read_csv(path)

    # strict: raises on bad dates, so pandas stops with an error instead of carrying on.
    df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
    df = df.sort_values("Date", kind="stable").reset_index(drop=True)

    return df

def summarise(ticker, df):
    dup_dates = df.loc[df["Date"].duplicated(keep=False), "Date"].unique()
    return {"ticker": ticker,
            "start": df["Date"].min().date(),
            "end": df["Date"].max().date(),
            "rows": len(df),
            "n_dup_dates": len(dup_dates)}

data = {t: load_etf(os.path.join(RAW, f)) for t, f in FILES.items()}
summary_df = pd.DataFrame([summarise(t, df) for t, df in data.items()]).set_index("ticker")
summary_df

,start,end,rows,n_dup_dates
ticker,,,,
SPY,2005-02-25,2017-11-10,3201,0
TLT,2005-02-25,2017-11-10,3201,0
XLF,2005-02-25,2017-11-10,3201,0


# Step 2: Clean and sanity-check prices

We drop the `OpenInt` column after confirming it is all zeros, since open interest has no meaning for an ETF. We then flag rows that are physically impossible: a High below the Low, a price of zero or less, or negative volume. These rows are reported, not deleted. Genuine extreme days, such as the 2010 flash crash, are kept, because they are exactly the high-volatility events we want to forecast.

In [3]:
PRICE_COLS = ["Open", "High", "Low", "Close"]

def clean_and_check(ticker, df):
    # 1. OpenInt: confirm it carries no information, then drop it
    assert (df["OpenInt"] == 0).all(), f"[{ticker}] OpenInt has non-zero values"
    df = df.drop(columns="OpenInt")

    # 2. Sanity checks: one boolean mask per rule (True = impossible row)
    checks = {
        "high_below_low": df["High"] < df["Low"],
        "price_not_positive": (df[PRICE_COLS] <= 0).any(axis=1),
        "negative_volume": df["Volume"] < 0,
    }

    # 3. Report, don't delete: print how many rows break each rule
    for rule, mask in checks.items():
        n_bad = int(mask.sum())
        print(f"[{ticker}] {rule}: {n_bad} rows")
        if n_bad:
            print("   dates:", df.loc[mask, "Date"].dt.strftime("%Y-%m-%d").tolist())

    return df

# Apply to the three tables from step 1
data = {t: clean_and_check(t, df) for t, df in data.items()}

[SPY] high_below_low: 0 rows
[SPY] price_not_positive: 0 rows
[SPY] negative_volume: 0 rows
[TLT] high_below_low: 0 rows
[TLT] price_not_positive: 0 rows
[TLT] negative_volume: 0 rows
[XLF] high_below_low: 0 rows
[XLF] price_not_positive: 0 rows
[XLF] negative_volume: 0 rows


# Step 3: Merge VIX and align to SPY trading days

We add the VIX (series VIXCLS from FRED) and combine all four files into one table with one row per SPY trading day. Where TLT, XLF or the VIX has no value for a SPY day, we carry the last known value forward, never backward, so no row ever uses information from the future. A before/after table shows the percentage of missing values in each column.

In [4]:
# 3a. Load the VIX from FRED; holidays are "." or blank, so read them as missing
vix = pd.read_csv(RAW + "VIXCLS.csv", na_values=".")
vix = vix.rename(columns={vix.columns[0]: "Date", "VIXCLS": "VIX"})  # date column name varies by download
vix["Date"] = pd.to_datetime(vix["Date"], format="%Y-%m-%d")

# 3b. Prefix TLT/XLF columns so they don't clash with SPY's (e.g. "TLT_Close")
def add_prefix(df, ticker):
    return df.rename(columns={c: f"{ticker}_{c}" for c in df.columns if c != "Date"})

tlt = add_prefix(data["TLT"], "TLT")
xlf = add_prefix(data["XLF"], "XLF")

# 3c. Start from SPY's dates and left-join the others onto them
merged = data["SPY"]
for other in [tlt, xlf, vix]:
    merged = merged.merge(other, on="Date", how="left", validate="one_to_one")
assert len(merged) == len(data["SPY"]), "Merge changed the number of SPY rows"

# 3d. % missing per column BEFORE filling
missing_before = merged.isna().mean() * 100

# 3e. Forward-fill only (never backward): copy the last known value onto gaps
fill_cols = [c for c in merged.columns if c.startswith(("TLT_", "XLF_")) or c == "VIX"]
merged[fill_cols] = merged[fill_cols].ffill()

# 3f. % missing AFTER, side by side with before
missing_after = merged.isna().mean() * 100
missing_table = pd.DataFrame({"before_%": missing_before, "after_%": missing_after}).round(2)
missing_table

,before_%,after_%
Date,0.0,0.0
Open,0.0,0.0
High,0.0,0.0
Low,0.0,0.0
Close,0.0,0.0
Volume,0.0,0.0
TLT_Open,0.0,0.0
TLT_High,0.0,0.0
TLT_Low,0.0,0.0
TLT_Close,0.0,0.0


# Step 4 — Compute returns, target and memory features (1–3)

We compute SPY's daily log returns and, from them, realised volatility (RV) over 1, 5 and 22 trading days: yesterday's shock, last week and last month. These three features form the HAR-RV benchmark. We also build the target: the log of realised volatility over the *next* 5 trading days. On 11 days (10 in training, 1 in test) SPY's close repeats exactly, giving a zero return whose log is −∞. We raise these to a floor equal to the smallest non-zero absolute return in the training period, computed on training data only, so no test information is used and no genuine trading day is altered. A final check confirms that only the zero-return days were changed.

In [5]:
import numpy as np

TRAIN_END = "2014-12-31"   # last training day (plan: train 2005-2014)

def log_returns(close):
    """Daily log return: r_t = ln(Close_t / Close_{t-1}). First row is NaN."""
    return np.log(close).diff()

def log_rv(r, n, floor):
    """Log realised volatility over the n days UP TO AND INCLUDING t (usable as a feature).
    RV = sqrt(sum of squared returns); n=1 gives |r_t|. Values below `floor` are raised to it."""
    rv = np.sqrt(r.pow(2).rolling(n).sum())
    return np.log(rv.clip(lower=floor))

def log_frv(r, n, floor):
    """Log realised volatility over the n days AFTER t (target only, never a feature)."""
    return log_rv(r, n, floor).shift(-n)

# 4a. SPY daily log returns
merged["r"] = log_returns(merged["Close"])

# 4b. Zero-return check: days where the close repeats exactly (prices have only 2-3 decimals)
is_zero  = merged["r"] == 0
is_train = merged["Date"] <= TRAIN_END
print(f"Zero-return days: {is_zero.sum()} (train: {(is_zero & is_train).sum()}, test: {(is_zero & ~is_train).sum()})")
print(merged.loc[is_zero, "Date"].dt.strftime("%Y-%m-%d").tolist())

# 4c. Floor = smallest NON-ZERO |r| in the training period (training only -> no test information).
#     Zero days get the smallest move we can actually measure; no real day is changed.
FLOOR = merged.loc[is_train & ~is_zero, "r"].abs().min()
print(f"Floor: {FLOOR:.2e} (log = {np.log(FLOOR):.2f})")

# 4d. Target: log RV over the next 5 trading days
merged["y"] = log_frv(merged["r"], 5, FLOOR)

# 4e. Features 1-3 (HAR memory): yesterday's shock, last week, last month
merged["f1"] = log_rv(merged["r"], 1, FLOOR)    # log RV_1 = log|r_t|
merged["f2"] = log_rv(merged["r"], 5, FLOOR)    # log RV_5
merged["f3"] = log_rv(merged["r"], 22, FLOOR)   # log RV_22

# 4f. Sanity check: the floor changed only the zero-return days
with np.errstate(divide="ignore"):
    changed = (merged["f1"] != np.log(merged["r"].abs())) & merged["r"].notna()
assert changed.sum() == is_zero.sum(), "Floor altered non-zero days"

Zero-return days: 11 (train: 10, test: 1)
['2005-08-22', '2005-11-08', '2007-12-19', '2012-07-30', '2012-10-31', '2013-03-27', '2013-06-10', '2013-07-30', '2013-10-21', '2014-02-26', '2017-01-10']
Floor: 3.27e-05 (log = -10.33)


# Step 5 — Build the leverage feature (5)

We add SPY's 1-day log return with its sign kept (f5 = rₜ). Feature 1 already measures the size of today's move; this feature adds its direction, because falls tend to raise future volatility more than rises of the same size (the leverage effect). It uses only day t's return, so it is known at time t and leaks nothing from the future. A quick check on the training period confirms the effect in our data: after the 10% biggest falls, next-week log volatility averages −3.44, against −3.70 after the 10% biggest rises and −4.03 overall.

In [6]:
# 5a. Feature 5 (leverage effect): today's return WITH its sign.
#     f1 already holds the SIZE of today's move (log|r_t|); f5 adds its DIRECTION.
merged["f5"] = merged["r"]

# 5b. Sanity check: f5 uses only day t's return (no shift), so it is known at t -> no leakage
assert merged["f5"].equals(merged["r"]), "f5 must equal today's return"

# 5c. Leverage-effect check, on TRAINING data only (the test set stays untouched until Sat 14 Nov):
#     average next-week log volatility after the biggest falls vs the biggest rises
train = merged[merged["Date"] <= TRAIN_END]
low, high = train["f5"].quantile([0.10, 0.90])
print(f"Mean y after the 10% biggest falls: {train.loc[train['f5'] <= low,  'y'].mean():.2f}")
print(f"Mean y after the 10% biggest rises: {train.loc[train['f5'] >= high, 'y'].mean():.2f}")
print(f"Mean y on all training days:        {train['y'].mean():.2f}")

Mean y after the 10% biggest falls: -3.44
Mean y after the 10% biggest rises: -3.70
Mean y on all training days:        -4.03


# Step 6 — Build VIX and cross-asset features (4, 6–8)

# Step 7 — Build activity and trend features (9–10)

# Step 8 — Trim incomplete rows, split and save model table

# Step 9 — Set up CV splitter, metrics and prediction export